# Laboratory 2 — Simulation and boundary conditions

**Competency:** C3 (Apply)

You will impose the clamped-edge condition and the uniform pressure load, run the solver, 
and extract the deflection field.

Two solver paths are available and they report the same quantities. Use Elmer if it is 
installed; otherwise use the pure-Python path, which needs nothing extra.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import numpy as np, matplotlib.pyplot as plt
from memslab.plate import Diaphragm, REFERENCE, REFERENCE_PRESSURE
from memslab.morley import solve_clamped_square

d, q = REFERENCE, REFERENCE_PRESSURE
result = solve_clamped_square(d, q, refine=5)

print(f'degrees of freedom : {result["ndof"]}')
print(f'peak deflection    : {result["w_max"]*1e6:.4f} um')
print(f'analytical         : {d.w_max(q)*1e6:.4f} um')
print(f'peak stress        : {result["sigma_max"]/1e6:.1f} MPa')


## What the boundary condition actually does

A clamped edge fixes two things: the deflection $w = 0$ and the slope $\partial w/\partial n = 0$. 
Fixing only the deflection gives a simply supported plate, which is a different problem with a 
noticeably larger deflection.

The Morley element carries the deflection at the vertices and the normal slope at the edge 
midpoints, so both conditions are imposed by setting every boundary degree of freedom to zero.


In [ ]:
# TASK: predict, before running anything, whether a simply supported plate deflects
# more or less than a clamped one, and by roughly how much.
# Then look up the simply supported coefficient and check your prediction.

# TASK: what happens to w_max if you double the pressure? If you double the thickness?
# Predict first from the algebra, then confirm numerically.

for factor in (0.5, 1.0, 2.0):
    print(f'q x {factor}: w_max = {d.w_max(q*factor)*1e6:.3f} um')


## Visualise the field


In [ ]:
from skfem import Basis, ElementTriP1
import numpy as np

mesh = result['mesh']
w = result['w'][result['basis'].nodal_dofs].flatten()

fig, ax = plt.subplots(figsize=(5, 4.2), dpi=120)
tri = ax.tricontourf(mesh.p[0]*1e6, mesh.p[1]*1e6, mesh.t.T, w*1e6, levels=20)
fig.colorbar(tri, ax=ax, label='deflection [um]')
ax.set_xlabel('x [um]'); ax.set_ylabel('y [um]'); ax.set_aspect('equal')
ax.set_title('Deflection under uniform pressure'); plt.show()


## Reflection

1. Your computed deflection does not equal the analytical value. Before Laboratory 3, write 
   down every reason you can think of for the difference.
2. Which of those reasons would get smaller with a finer mesh, and which would not?
